In [5]:
import os
import json
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import BlipProcessor, BlipForConditionalGeneration

# Import the standard evaluation metrics
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.meteor.meteor import Meteor


CHECKPOINT_DIR = "./blip_video_model_2"
CSV_PATH = "./msrvtt_2k_preprocessed.csv"
NPZ_DIR = "./final_frames_v2"
OUTPUT_REPORT = "final_evaluation_report.txt"

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

print(f"Loading Final Model from {CHECKPOINT_DIR} for Comprehensive Evaluation...")
processor = BlipProcessor.from_pretrained(CHECKPOINT_DIR)
model = BlipForConditionalGeneration.from_pretrained(CHECKPOINT_DIR)
model.to(DEVICE)
model.eval()


print("Grouping ground-truth human captions...")
df = pd.read_csv(CSV_PATH)

gts = {}
for _, row in df.iterrows():
    vid = row['video_id']
    cap = str(row['caption'])
    if vid not in gts:
        gts[vid] = []
    gts[vid].append(cap)

unique_videos = list(gts.keys())
print(f"Found {len(unique_videos)} unique videos to evaluate.")


res = {}

print("Generating AI captions for scoring...")
for vid in tqdm(unique_videos, desc="Evaluating Videos"):
    npz_path = os.path.join(NPZ_DIR, f"{vid}.npz")
    
    try:
        with np.load(npz_path) as data:
            video = data['video'] 
            
        # Extract middle frame
        middle_idx = video.shape[1] // 2
        frame = video[:, middle_idx, :, :] 
        
        # Format for BLIP
        frame = np.transpose(frame, (1, 2, 0))
        if frame.max() <= 1.0:
            frame = (frame * 255).astype(np.uint8)
        else:
            frame = frame.astype(np.uint8)
            
        inputs = processor(images=frame, return_tensors="pt").to(DEVICE)
        
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_length=30,
                num_beams=5,
                early_stopping=True,
                no_repeat_ngram_size=2
            )
            
        generated_caption = processor.decode(output_ids[0], skip_special_tokens=True)
        res[vid] = [generated_caption]
        
    except Exception as e:
        print(f"Error processing {vid}: {e}")
        continue


print("\nCalculating metrics...")

# Initialize scorers
scorers = [
    (Bleu(4), ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4"]),
    (Meteor(), "METEOR"),
    (Rouge(), "ROUGE_L"),
    (Cider(), "CIDEr")
]

final_scores = {}

for scorer, method in scorers:
    print(f"Computing {method}...")
    try:
        score, scores = scorer.compute_score(gts, res)
        if type(method) == list:
            for sc, scs, m in zip(score, scores, method):
                final_scores[m] = sc * 100 # Multiply by 100 for standard paper format
        else:
            final_scores[method] = score * 100
    except Exception as e:
        print(f"Warning: Failed to compute {method}. Error: {e}")
        final_scores[method] = "Error computing metric"


print("\n" + "="*40)
print("🏆 FINAL EVALUATION REPORT")
print("="*40)
for metric, val in final_scores.items():
    if isinstance(val, float):
        print(f"{metric}: {val:.2f}")
    else:
        print(f"{metric}: {val}")
print("="*40)

# Write to text file
with open(OUTPUT_REPORT, "w") as f:
    f.write("FINAL MODEL EVALUATION REPORT\n")
    f.write(f"Model: {CHECKPOINT_DIR}\n")
    f.write("="*40 + "\n")
    for metric, val in final_scores.items():
        if isinstance(val, float):
            f.write(f"{metric}: {val:.2f}\n")
        else:
            f.write(f"{metric}: {val}\n")
    f.write("="*40 + "\n")

print(f"\n✅ Report successfully saved to: {OUTPUT_REPORT}")

Loading Final Model from ./blip_video_model_2 for Comprehensive Evaluation...


Loading weights: 100%|██████████| 473/473 [00:00<00:00, 7911.70it/s]
The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Grouping ground-truth human captions...
Found 2000 unique videos to evaluate.
Generating AI captions for scoring...


Evaluating Videos: 100%|██████████| 2000/2000 [12:50<00:00,  2.60it/s]



Calculating metrics...
Computing ['Bleu_1', 'Bleu_2', 'Bleu_3', 'Bleu_4']...
{'testlen': 15514, 'reflen': 15707, 'guess': [15514, 13514, 11514, 9514], 'correct': [14059, 10051, 6647, 4046]}
ratio: 0.9877124848792902
Computing METEOR...
Computing ROUGE_L...
Computing CIDEr...

🏆 FINAL EVALUATION REPORT
Bleu_1: 89.50
Bleu_2: 81.08
Bleu_3: 72.10
Bleu_4: 62.99
METEOR: Error computing metric
ROUGE_L: 73.08
CIDEr: 87.39

✅ Report successfully saved to: final_evaluation_report.txt
